<a href="https://colab.research.google.com/github/HKozuka/IDX-Exchange-Data-Science/blob/main/03_baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IDX Exchange - Baseline Model**

## Setup
#### Mount Google Drive & Load the Validation Fold

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import pandas as pd
import numpy as np

file_path = '/content/drive/MyDrive/IDX Exchange Data Science Internship/Google Colab Notebooks/validation_fold_1.csv'

df_val_fold1 = pd.read_csv(file_path)
df_val_fold1.head()

,log_ClosePrice,CloseDate,ViewYN,WaterfrontYN,PoolPrivateYN,Stories,FireplaceYN,log_LivingArea,BathroomsTotalInteger,log_BuildingAreaTotal,BedroomsTotal,AssociationFee,log_LotSizeSquareFeet,GarageSpaces,AssociationFeeFrequency_Monthly,AssociationFeeFrequency_None,AssociationFeeFrequency_Quarterly,AssociationFeeFrequency_SemiAnnually
0,13.864301,2024-05-17,0,0,0,1.0,1,7.433667,2.0,7.601402,4.0,0.0,8.881975,2.0,0,1,0,0
1,13.976779,2024-05-17,1,0,0,1.0,1,7.835579,3.0,7.601402,4.0,85.0,10.325252,3.0,1,0,0,0
2,14.272052,2024-05-17,0,0,1,1.0,1,7.437795,2.0,7.601402,4.0,0.0,8.865594,2.0,0,1,0,0
3,13.693343,2024-05-17,0,0,0,1.0,1,7.131699,2.0,7.601402,2.0,425.0,7.131699,1.0,1,0,0,0
4,13.060488,2024-05-17,1,0,0,2.0,0,7.560080,3.0,7.601402,4.0,140.0,7.771489,2.0,1,0,0,0


## Train baseline Linear Regression Model
Note: We will split the features and target. Please modify the column names in the code below if they differ in your dataset.

In [33]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

y = df_val_fold1['log_ClosePrice']
X = df_val_fold1.drop(columns=['log_ClosePrice', 'CloseDate'])

# Instantiate and fit Linear Regression model!
model = LinearRegression()
model.fit(X, y)

# Predict and Evaluate
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)
mse = mean_squared_error(y, y_pred)

print(f"Baseline Linear Regression Metrics on Validation Data Fold 1:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared Score (R2): {r2:.4f}")

Baseline Linear Regression Metrics on Validation Data Fold 1:
Mean Squared Error (MSE): 0.2538
Mean Absolute Error (MAE): 0.4054
R-squared Score (R2): 0.3390


## Evaluate Model on Test Dataset

In [34]:
# Load the test dataset
test_file_path = '/content/drive/MyDrive/IDX Exchange Data Science Internship/Google Colab Notebooks/test_dataset.csv'
df_test = pd.read_csv(test_file_path)

y_test_actual = df_test['log_ClosePrice']
X_test_features = df_test.drop(columns=['log_ClosePrice', 'CloseDate'])

# Predict
y_test_pred = model.predict(X_test_features)

# Eval metrics
test_mse = mean_squared_error(y_test_actual, y_test_pred)
test_mae = mean_absolute_error(y_test_actual, y_test_pred)
test_r2 = r2_score(y_test_actual, y_test_pred)

print(f"Baseline Linear Regression Metrics on Test Dataset:")
print(f"Mean Squared Error (MSE): {test_mse:.4f}")
print(f"Mean Absolute Error (MAE): {test_mae:.4f}")
print(f"R-squared Score (R2): {test_r2:.4f}")

Baseline Linear Regression Metrics on Test Dataset:
Mean Squared Error (MSE): 0.2624
Mean Absolute Error (MAE): 0.4115
R-squared Score (R2): 0.3440


In [35]:
# Calculate real-world dollar values from log-transformed predictions
y_test_actual_usd = np.exp(y_test_actual)
y_test_pred_usd = np.exp(y_test_pred)

# Calculate how many predictions are within $10,000 of the true price
dollar_errors = np.abs(y_test_actual_usd - y_test_pred_usd)
within_10k_count = np.sum(dollar_errors <= 10000)
accuracy_within_10k = (within_10k_count / len(df_test)) * 100

print(f"Baseline Model Performance in Real USD on Test Dataset:")
print(f"Total Test Samples: {len(df_test):,}")
print(f"Predictions within +/- $10,000 of actual price: {within_10k_count:,} ({accuracy_within_10k:.2f}%)")

Baseline Model Performance in Real USD on Test Dataset:
Total Test Samples: 12,405
Predictions within +/- $10,000 of actual price: 201 (1.62%)


### **Summary of Baseline Model Results (Extremely Poor Performance)**

* **Model Type:** Baseline Linear Regression (trained on log-transformed close prices, `log_ClosePrice`).

| Dataset | Mean Squared Error (MSE) | Mean Absolute Error (MAE) | $R^2$ Score |
| :--- | :---: | :---: | :---: |
| **Validation Fold 1** (N=49,219) | `0.2538` | `0.4054` | `0.3390` |
| **Test Dataset** (N=12,405) | `0.2624` | `0.4115` | `0.3440` |

#### **Key Insights & Real-World Limitations (Why this is super bad):**
* **Extremely Low Accuracy in Real USD:** When back-transforming predictions to actual dollar values ($e^{y}$), **only 1.62% of predictions (201 out of 12,405)** fall within a realistic, practical window of **$\pm\$10,000$** from the true house price.
* **Unusable for Production:** An AVM (Automated Valuation Model) that is off by more than $\$10,000$ on **98.38%** of properties is highly inaccurate and completely unusable for real-world business decisions.
* **Next Steps:** We must move away from simple linear models on basic features and introduce more robust non-linear architectures to capture the actual underlying patterns!